Preparing the CFPB (Consumer Financial Protection Bureau) Customer Complaints Database for sentiment analysis requires handling unique, heavily redacted textual data (Consumer complaint narrative) alongside structured categories.

## Loading dataset 

In [2]:
import argparse
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

def read_with_pandas(path, out=None):
    import pandas as pd

    df = pd.read_csv(path)
    print(f"Read {len(df)} rows, {len(df.columns)} columns")
    print(df.head().to_string(index=False))

    if out:
        save_df(df, out)
    return df


def save_df(df, out_path):
    """Save DataFrame to out_path; choose method by extension."""
    ext = os.path.splitext(out_path)[1].lower()
    try:
        if ext in ('.pkl', '.pickle'):
            df.to_pickle(out_path)
        elif ext in ('.parquet', '.pq'):
            df.to_parquet(out_path)
        elif ext in ('.feather', '.fth'):
            df.to_feather(out_path)
        elif ext in ('.csv', '.txt'):
            df.to_csv(out_path, index=False)
        else:
            raise ValueError(f'Unsupported output extension: {ext}')
        print(f'Saved DataFrame to {out_path}')
    except Exception as e:
        print(f'Failed to save DataFrame: {e}', file=sys.stderr)
        raise


In [3]:
#run scriopt under terminal command to read complaints_in_just_2025.csv and show a preview
#%run -i read_complaints.py --path ~/Documents/capstone_project/complaints_in_just_2025.csv --out complaints_2025.csv

In [4]:
# load cfpb 2025 complaints data
df = read_with_pandas('complaints_in_just_2025.csv', out = None)

Read 5443005 rows, 16 columns
Date received                                             Product                    Sub-product                                Issue                                            Sub-issue                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [5]:
df.head(5)

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2025-01-09,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,NaN,MOUNTAIN AMERICA FEDERAL CREDIT UNION,AZ,85301,NaN,Web,2025-01-27,Closed with explanation,Yes,11454251
1,2025-04-08,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,"To Whom It May Concern, I am filing a formal c...",NaN,MOHELA,NE,681XX,NaN,Web,2025-04-08,Closed with explanation,No,12856469
2,2025-04-15,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",Struggling with a XXXX has significantly impac...,NaN,"Maximus Federal Services, Inc.",AL,35007,NaN,Web,2026-04-06,Closed with explanation,Yes,12989708
3,2025-04-18,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Trouble with how payments are being handled,I have taken out parent loans on behalf of my ...,NaN,MOHELA,OH,43110,Servicemember,Web,2025-04-18,Closed with explanation,No,13062648
4,2025-05-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,NaN,NaN,MOHELA,OH,44224,NaN,Web,2025-05-05,Closed with explanation,No,13330457


In [6]:
# The 2025 dataset has 5,443,005 rows and 16 columns
# The full dataset has 15,896,496 observations
df.shape

(5443005, 16)

In [7]:
# List the raw data types of all columns of the Pandas data frame
df.dtypes

Date received                     str
Product                           str
Sub-product                       str
Issue                             str
Sub-issue                         str
Consumer complaint narrative      str
Company public response           str
Company                           str
State                             str
ZIP code                          str
Tags                              str
Submitted via                     str
Date sent to company              str
Company response to consumer      str
Timely response?                  str
Complaint ID                    int64
dtype: object

In [8]:
# Convert the date columns from a string object format to a datetime format and then normalize the date to midnight, i.e. YYYY-MM-DD

df['Date received'] = pd.to_datetime(df['Date received'].str[:10]).dt.normalize()
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'].str[:10]).dt.normalize()

In [9]:
# Look at a few obs to make sure we have correctly formatformatted date and time.
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2025-01-09,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,NaN,MOUNTAIN AMERICA FEDERAL CREDIT UNION,AZ,85301,NaN,Web,2025-01-27,Closed with explanation,Yes,11454251
1,2025-04-08,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,"To Whom It May Concern, I am filing a formal c...",NaN,MOHELA,NE,681XX,NaN,Web,2025-04-08,Closed with explanation,No,12856469
2,2025-04-15,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",Struggling with a XXXX has significantly impac...,NaN,"Maximus Federal Services, Inc.",AL,35007,NaN,Web,2026-04-06,Closed with explanation,Yes,12989708
3,2025-04-18,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Trouble with how payments are being handled,I have taken out parent loans on behalf of my ...,NaN,MOHELA,OH,43110,Servicemember,Web,2025-04-18,Closed with explanation,No,13062648
4,2025-05-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,NaN,NaN,MOHELA,OH,44224,NaN,Web,2025-05-05,Closed with explanation,No,13330457


In [10]:
# Group complaints by year received and count observations
df.groupby(df['Date received'].dt.year).size()

Date received
2025    5443005
dtype: int64

In [11]:
# Group complaints by product category and count observations
df.groupby(df['Product']).size()

Product
Checking or savings account                                  84198
Credit card                                                  89989
Credit reporting or other personal consumer reports        4810314
Debt collection                                             283206
Debt or credit management                                     4267
Money transfer, virtual currency, or money service           84618
Mortgage                                                     24694
Payday loan, title loan, personal loan, or advance loan      13135
Prepaid card                                                  6499
Student loan                                                 21314
Vehicle loan or lease                                        20771
dtype: int64

In [12]:
# Group complaints by issue category and count observations

df.groupby(df['Issue']).size()

Issue
Advertising                                                        99
Advertising and marketing, including promotional offers          3106
Applying for a mortgage or refinancing an existing mortgage      2596
Attempts to collect debt not owed                              115683
Can't contact lender or servicer                                  203
                                                                ...  
Vehicle was repossessed or sold the vehicle                       149
Was approved for a loan, but didn't receive money                  23
Was approved for a loan, but didn't receive the money             105
Written notification about debt                                 64048
Wrong amount charged or received                                  459
Length: 90, dtype: int64

In [13]:
# Group by the company public response and count obsevations

df.groupby(df['Company public response']).size()

Company public response
Company believes complaint caused principally by actions of third party outside the control or direction of the company        555
Company believes complaint is the result of an isolated error                                                                  378
Company believes complaint relates to a discontinued policy or procedure                                                         8
Company believes complaint represents an opportunity for improvement to better serve consumers                                 596
Company believes it acted appropriately as authorized by contract or law                                                     48420
Company believes the complaint is the result of a misunderstanding                                                             581
Company believes the complaint provided an opportunity to answer consumer's questions                                         2172
Company can't verify or dispute the facts in the complaint 

In [14]:
# Group by company response to consumer and count obsevations

df.groupby(df['Company response to consumer']).size()

Company response to consumer
Closed with explanation            3194212
Closed with monetary relief          26109
Closed with non-monetary relief    2212222
In progress                            365
Untimely response                    10097
dtype: int64

In [15]:
# Group by timely response to consumer complaints and count obsevations

df.groupby(df['Timely response?']).size()

Timely response?
No       24595
Yes    5418410
dtype: int64

In [16]:
# Count up the number of unique companies having consumer complaints

df['Company'].nunique()

3961

In [17]:
# Find the top 10 companies with the most consumer complaints
df['Company'].value_counts().head(10)

Company
EQUIFAX, INC.                             1680538
TRANSUNION INTERMEDIATE HOLDINGS, INC.    1601851
Experian Information Solutions Inc.       1388821
Block, Inc.                                 48091
CAPITAL ONE FINANCIAL CORPORATION           34102
CBC Companies, Inc.                         30769
Early Warning Services, LLC                 24509
JPMORGAN CHASE & CO.                        24458
Resurgent Capital Services L.P.             23741
CITIBANK, N.A.                              19791
Name: count, dtype: int64

In [18]:
df_2025 =df.copy

## Data preparation
The CFPB database is massive, but not all entries contain text because consumers must explicitly opt-in to share their narratives.

Here is a step-by-step pipeline to clean, preprocess, and structure this data for Natural Language Processing (NLP).

* Filter out rows where Consumer complaint narrative is null.

* The CFPB heavily sanitizes data for privacy, replacing names, account numbers, and dates with blocks of XXXX. They hold no sentiment 
value and can confuse word embeddings. Therefore, remove them or replace them with a single space. 

In [19]:
#load modules
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [20]:
df = df.dropna(subset=['Consumer complaint narrative'])
# Rename for ease
df['narrative'] = df['Consumer complaint narrative']

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,narrative
1,2025-04-08,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,"To Whom It May Concern, I am filing a formal c...",NaN,MOHELA,NE,681XX,NaN,Web,2025-04-08,Closed with explanation,No,12856469,"To Whom It May Concern, I am filing a formal c..."
2,2025-04-15,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",Struggling with a XXXX has significantly impac...,NaN,"Maximus Federal Services, Inc.",AL,35007,NaN,Web,2026-04-06,Closed with explanation,Yes,12989708,Struggling with a XXXX has significantly impac...
3,2025-04-18,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Trouble with how payments are being handled,I have taken out parent loans on behalf of my ...,NaN,MOHELA,OH,43110,Servicemember,Web,2025-04-18,Closed with explanation,No,13062648,I have taken out parent loans on behalf of my ...
6,2025-05-21,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,On my XXXX credit report it has my student lo...,NaN,MOHELA,CO,80020,NaN,Web,2025-05-21,Closed with explanation,No,13618226,On my XXXX credit report it has my student lo...
7,2025-05-29,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,I XXXX XXXX am writing to tell you about whats...,NaN,MOHELA,MI,48228,Servicemember,Web,2025-05-29,Closed with explanation,No,13768659,I XXXX XXXX am writing to tell you about whats...


In [21]:
# Noticing lots of customers didn't provide their narrative with their complaint, so we will drop those rows from the dataset.
df.shape

(1221970, 17)

## Text Preprocessing Pipeline 
For Machine Learning (TF-IDF + Logistic Regression) or Modern Deep Learning require clean, normalized text to reduce vocabulary size.

Belor are pipelines to pre-process text data:

* Convert all text to lowercase.  
* Remove Punctuation & Special Characters to liminate noise.  
* Remove/Strip out the XXXX strings.
* Tokenization & Stop Words: Split text into words and remove words that carry no meaning, but still keeping negation words like "not" or "no", as they completely flip sentiment.
* Lemmatization: Reduce words to their base form (e.g., "charged", "charging", "charges" to  their based word "charge").  

Moreover, we can also remove url in the text or remove dupliucate line or empty/whiltespace lines.

In [24]:
def normalize_masks(text):
    # Convert date masks to a standard token
    text = re.sub(r'\b[xX]{2}/[xX]{2}/[xX]{4}\b', '[MASKED_DATE]', text)
    # Convert dollar/numeric masks
    text = re.sub(r'\$[xX,]+', '[MASKED_AMOUNT]', text)
    # Convert remaining generic masks
    text = re.sub(r'[xX]{2,}', '[MASKED_TEXT]', text)
    # Standardize the CFPB masking
    text = re.sub(r'X{2,}', '[REDACTED]', text)
    return text

df['cleaned_narrative'] = df['narrative'].apply(normalize_masks)


In [25]:
# Filter out short, non-meaningful descriptions (under 20 words)
df = df[df['cleaned_narrative'].apply(lambda x: len(str(x).split()) > 20)]

In [26]:
import html

def clean_web_artifacts(text):
    text = html.unescape(text) # Converts &amp; to &
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Removes URLs
    text = re.sub(r'\s+', ' ', text).strip() # Removes duplicate whitespace/newlines
    return text

In [27]:
df['cleaned_narrative'] = df['cleaned_narrative'].apply(clean_web_artifacts)

#### Grouping shifting categorical targets to ensure text consistency

In [28]:

product_mapping = {
    'Credit card': 'Credit Card/Prepaid',
    'Prepaid card': 'Credit Card/Prepaid',
    'Credit card or prepaid card': 'Credit Card/Prepaid'
}
df['Product_Standardized'] = df['Product'].replace(product_mapping)

#### Keep only rows where a commercial relationship was verified and sent to the company

In [29]:

df = df[df['Submitted via'] != 'Referral'] 
df = df.dropna(subset=['Company response to consumer'])

#### Prune Document Length Extremes
Extremely long documents dilate the computational matrix size (especially for deep learning embedding vectors) and dilute the actual "sentiment" signal with thousands of words of standard chronological noise.

In [30]:
# Filter by token-count percentiles
df['word_count'] = df['cleaned_narrative'].apply(lambda x: len(x.split()))
min_thresh = df['word_count'].quantile(0.05)
max_thresh = df['word_count'].quantile(0.95)

df_filtered = df[(df['word_count'] >= min_thresh) & (df['word_count'] <= max_thresh)]

In [31]:
df_filtered.shape

(1055842, 20)

In [32]:
# Map responses to Binary Sentiment/Outcome (1 = Relief/Positive resolution, 0 = No relief)
relief_responses = ['Closed with monetary relief', 'Closed with non-monetary relief']
df_filtered['sentiment_label'] = df['Company response to consumer'].apply(lambda x: 1 if x in relief_responses else 0)

In [36]:
df_filtered.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,...,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,narrative,cleaned_narrative,Product_Standardized,word_count,sentiment_label
1,2025-04-08,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,"To Whom It May Concern, I am filing a formal c...",NaN,MOHELA,NE,681XX,...,Web,2025-04-08,Closed with explanation,No,12856469,"To Whom It May Concern, I am filing a formal c...","To Whom It May Concern, I am filing a formal c...",Credit reporting or other personal consumer re...,173,0
2,2025-04-15,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",Struggling with a XXXX has significantly impac...,NaN,"Maximus Federal Services, Inc.",AL,35007,...,Web,2026-04-06,Closed with explanation,Yes,12989708,Struggling with a XXXX has significantly impac...,Struggling with a [MASKED_TEXT] has significan...,Student loan,98,0
3,2025-04-18,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Trouble with how payments are being handled,I have taken out parent loans on behalf of my ...,NaN,MOHELA,OH,43110,...,Web,2025-04-18,Closed with explanation,No,13062648,I have taken out parent loans on behalf of my ...,I have taken out parent loans on behalf of my ...,Student loan,292,0
7,2025-05-29,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,I XXXX XXXX am writing to tell you about whats...,NaN,MOHELA,MI,48228,...,Web,2025-05-29,Closed with explanation,No,13768659,I XXXX XXXX am writing to tell you about whats...,I [MASKED_TEXT] [MASKED_TEXT] am writing to te...,Credit reporting or other personal consumer re...,356,0
10,2025-06-12,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Trouble with how payments are being handled,In XX/XX/XXXX and XX/XX/XXXX my employer made ...,NaN,MOHELA,MD,207XX,...,Web,2025-06-12,Closed with explanation,No,14034568,In XX/XX/XXXX and XX/XX/XXXX my employer made ...,In [MASKED_DATE] and [MASKED_DATE] my employer...,Student loan,301,0


In [33]:
from sklearn.model_selection import train_test_split

# Split data into Features (X) and Target Label (y)
X = df_filtered['cleaned_narrative']
y = df_filtered['sentiment_label']

# 80% Training, 20% Testing with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y,       # Crucial for maintaining class balance
    random_state=42     # Ensures reproducibility
)

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF automatically tokenizes your cleaned_narrative string behind the scenes
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X = vectorizer.fit_transform(df_filtered['cleaned_narrative'])

In [ ]:
df_filtered.to_csv('df_filtetred.csv', index=False)